# 第 3 章：从候选召回到可信上下文

**本章目标**：区分向量、词法与结构化检索；手算倒数排名融合（RRF, Reciprocal Rank Fusion）；理解重排序、依赖补全、过滤与评估。所有单元默认离线运行，不调用嵌入模型或数据库；可选精排实验需显式开启。

检索增强生成（RAG, Retrieval-Augmented Generation）的第一步是让必要证据**进入候选集合**。若“净销售额”的业务口径没被召回，后面的模型再会写结构化查询语言（SQL, Structured Query Language）也可能算错。若候选正确但排序靠后，融合与重排序才有施展空间。

## 3.1 两条召回通道解决不同问题

项目的生产检索使用两条通道：pgvector 的余弦距离找语义近邻；Python 中的 <code>lexical_score()</code> 计算显式业务别名和 <code>object_ref</code> 片段的命中分。每条通道最多取 15 个候选，再用 RRF 融合。词法得分是人为排序信号，不是概率，不能直接与余弦距离相加。

| 用户问法 | 向量通道可能擅长 | 精确匹配可能擅长 |
|---|---|---|
| “扣掉退款后的营收” | 找到意思接近的净销售额卡 | 如果运营维护了同义别名，也能稳定命中 |
| “净销售额” | 可能找到多个销售相关定义 | 显式别名准确命中 |
| “订单 000123 状态” | 编号容易被语义表示弱化 | 应路由到数据库精确过滤 |

**实现边界**：当前 <code>lexical_score()</code> 只检查 <code>aliases</code> 和 <code>object_ref</code> 拆分后的片段；它**不对正文中的字段清单做全文匹配**。例如输入 <code>paid_at</code>，现有词法通道并不能仅凭支付表正文里有这个字段就命中它。字段级检索需要另建字段索引或把关键字段维护为显式别名。

## 3.2 在项目知识卡上运行真实词法打分

下面读取本地知识卡并调用 <code>src/nl2sql/retrieval.py</code> 的真实函数。导入模块需要项目 Python 依赖已安装，但不会请求网络、读取密钥或连接数据库。

In [1]:
from pathlib import Path
import json
import sys

root = Path.cwd()
if root.name == "tutorial":
    root = root.parent
root = root.resolve()
sys.path.insert(0, str(root / "src"))
from nl2sql.retrieval import RetrievedDoc, lexical_score, rrf_rank

raw = json.loads((root / "db" / "knowledge.json").read_text(encoding="utf-8"))
docs = {
    item["id"]: RetrievedDoc(
        doc_id=item["id"], kind=item["kind"], object_ref=item["object_ref"],
        content=item["content"], aliases=item["aliases"],
        related_tables=item["related_tables"],
    )
    for item in raw
}
question = "2026 年 8 月华东地区的净销售额是多少？"
lexical = sorted(
    ((doc_id, lexical_score(question, doc)) for doc_id, doc in docs.items()),
    key=lambda pair: (-pair[1], pair[0]),
)
lexical_rank = [doc_id for doc_id, score in lexical if score > 0]
print("词法候选：", [(doc_id, score) for doc_id, score in lexical if score > 0])
print("词法前五：", lexical_rank[:5])

词法候选： [('metric.net_sales', 7), ('metric.gross_sales', 6), ('table.customers', 5)]
词法前五： ['metric.net_sales', 'metric.gross_sales', 'table.customers']


In [2]:
payment_card = docs["table.payments"]
print("只输入 paid_at 的词法分：", lexical_score("paid_at", payment_card))
print("输入 payments.paid_at 的词法分：", lexical_score("payments.paid_at", payment_card))
assert lexical_score("paid_at", payment_card) == 0
print("后一问即使命中，也来自 object_ref 中的 payments，而非字段 paid_at 的全文匹配。")

只输入 paid_at 的词法分： 0
输入 payments.paid_at 的词法分： 4
后一问即使命中，也来自 object_ref 中的 payments，而非字段 paid_at 的全文匹配。


另一类常见词法方法是 BM25（Best Matching 25）：它使用词频、逆文档频率（IDF, Inverse Document Frequency）和长度归一化，对大量文本的关键词检索很有效。项目**没有使用 BM25**；13 张已审核的业务卡暂时使用可解释的别名规则。若资料变成数万份手册与工单，可以对照专业搜索引擎或 PostgreSQL 全文检索的 BM25 方案，并特别测试中文分词、实体名和字段名。

常见形式为：

$$\operatorname{BM25}(q,d)=\sum_{t\in q}\operatorname{IDF}(t)
\frac{tf_{t,d}(k_1+1)}
{tf_{t,d}+k_1(1-b+b|d|/\overline{|d|})}.$$

直觉是：匹配词越多通常越相关，但重复出现的收益逐渐变小；罕见词更有区分力；长文档不会仅因文字多就占便宜。公式中 <code>tf</code> 是词频（TF, Term Frequency），即当前词在文档中出现的次数；IDF 衡量该词在整个语料中的稀有程度；<code>|d|</code> 是文档长度。这里的词必须先正确分出来，公式本身不会替你解决中文分词。

## 3.3 融合：不用强行统一两种分数

项目采用的 RRF 只看每个通道的**名次**，不比较原始分数：

$$\operatorname{RRF}(d)=\sum_{j \in \{\text{向量},\text{词法}\}}
\frac{1}{k+\operatorname{rank}_j(d)}.$$

未进入某通道的文档，该通道贡献 0。项目设置 <code>k=60</code>，名次从 1 开始。<code>k</code> 是平滑常数，不是候选数。两条通道都靠前的文档会得到两份贡献。RRF 不会把两条都错的召回变成正确证据。

In [3]:
# 向量名次是离线模拟；词法名次来自上面的真实项目函数。
vector_rank = [
    "metric.net_sales", "table.payments", "rule.time",
    "join.customer_orders", "table.refunds",
]
fused = rrf_rank(vector_rank, lexical_rank)
for doc_id in fused[:7]:
    parts = []
    for ranking in (vector_rank, lexical_rank):
        parts.append(1 / (60 + ranking.index(doc_id) + 1) if doc_id in ranking else 0)
    print(f"{doc_id:28} 向量={parts[0]:.5f} 词法={parts[1]:.5f} 合计={sum(parts):.5f}")
print("融合前七：", fused[:7])

metric.net_sales             向量=0.01639 词法=0.01639 合计=0.03279
metric.gross_sales           向量=0.00000 词法=0.01613 合计=0.01613
table.payments               向量=0.01613 词法=0.00000 合计=0.01613
rule.time                    向量=0.01587 词法=0.00000 合计=0.01587
table.customers              向量=0.00000 词法=0.01587 合计=0.01587
join.customer_orders         向量=0.01562 词法=0.00000 合计=0.01562
table.refunds                向量=0.01538 词法=0.00000 合计=0.01538
融合前七： ['metric.net_sales', 'metric.gross_sales', 'table.payments', 'rule.time', 'table.customers', 'join.customer_orders', 'table.refunds']


## 3.4 候选之后还要做什么

**重排序（Rerank）**让模型同时读取问题和少量候选原文，对细粒度相关性重新判断。它可能改善候选顺序，但不能找回首轮完全漏掉的卡。额外的模型调用还会增加延迟和费用。当前智能体（Agent）的生产路径到 RRF 为止；项目有独立的 Rerank 试验代码，不能把它当成已上线步骤。

本项目还有一个关键动作：若选中了 <code>metric.net_sales</code>，依据它的 <code>related_tables</code> 补齐支付、退款和订单表卡。下面只模拟这一步，不调用生产 <code>retrieve()</code>，因为后者会请求真实问题向量并读数据库。

In [4]:
selected = ["metric.net_sales"]
table_ids = sorted({
    f"table.{table}"
    for doc_id in selected
    for table in docs[doc_id].related_tables
})
expanded = selected + [doc_id for doc_id in table_ids if doc_id in docs and doc_id not in selected]
context = "\n\n".join(f"[{doc_id}] {docs[doc_id].content}" for doc_id in expanded)
print("依赖补全：", expanded)
print("上下文字符数：", len(context))
assert {"table.orders", "table.payments", "table.refunds"} <= set(expanded)

依赖补全： ['metric.net_sales', 'table.orders', 'table.payments', 'table.refunds']
上下文字符数： 588


上下文要保留稳定文档标识符（ID, Identifier）、来源和版本，以便回答能回指证据。当前生产代码对拼接结果设 10,000 **字符**上限；字符数不等于模型词元（token）数。生产中通常按词元计预算，并优先保留指标、时间和连接规则，避免把关键否定词截掉。

### 可选在线对照：Qwen 精排

下一个单元默认跳过。只有把开关改为 <code>True</code> 才会调用 Qwen 的应用程序编程接口（API, Application Programming Interface），需要已配置密钥且业务空间开通重排序模型；该调用可能产生费用。实验只发送融合前 5 个候选的正文，比较名次变化，**不会修改智能体的生产检索路径**。若模型不可用，保留上面的 RRF 结果继续学习即可。

In [5]:
RUN_ONLINE_RERANK = False  # 显式改为 True 才会调用外部模型
if RUN_ONLINE_RERANK:
    from nl2sql.rerank import rerank_documents

    candidate_ids = fused[:5]
    payload = [(doc_id, docs[doc_id].content) for doc_id in candidate_ids]
    try:
        online_rank = rerank_documents(question, payload)
        print("RRF 前五：", candidate_ids)
        print("Qwen 精排：", [(doc_id, round(score, 4)) for doc_id, score in online_rank])
    except (RuntimeError, ValueError) as exc:
        print("可选实验未完成：", str(exc))
else:
    print("已跳过在线精排；默认离线运行不会调用 Qwen。")

已跳过在线精排；默认离线运行不会调用 Qwen。


## 3.5 查询改写、拆解和路由

原始用户输入未必是好检索查询。“那华南呢？”缺少前一轮指标与月份；“比较两地净销售额并解释退款差异”可能需要多个子问题。改写必须保留时间、地区、状态和否定词。下面是工程上常见的选项：

| 方案 | 解决的失败 | 代价与风险 |
|---|---|---|
| 规则式补全追问 | 明确的省略问法 | 覆盖范围有限 |
| 大语言模型（LLM, Large Language Model）多查询改写 | 同义问法与长问题漏召回 | 多次调用；可能改错过滤条件 |
| 假设文档嵌入（HyDE, Hypothetical Document Embeddings） | 短问题难匹配长文档 | 假设文本可能编造，只能用于召回 |
| 查询拆解与路由 | 多来源、多指标、多步计算 | 路由错会漏证据，子结果要核对 |
| 元数据过滤 | 租户、权限、有效期 | 必须在召回前或过程中执行 |

例如想问具体订单状态，应路由到结构化数据库查询；想问“净销售额如何定义”，先查知识。权限过滤不能等未授权文本进入模型后再做。当前教学库没有租户权限字段，这一行是扩展设计。

In [6]:
previous = {
    "month": "2026-08", "metric": "订单数",
    "region": "华东", "exclude_cancelled": True,
}
follow_up = "那华南呢？"
rewritten = (
    f"{previous['month']} 华南地区的{previous['metric']}是多少？"
    + ("排除取消订单。" if previous["exclude_cancelled"] else "")
)
print("补全后的查询：", rewritten)
assert all(term in rewritten for term in ("2026-08", "华南", "订单数", "排除取消订单"))

补全后的查询： 2026-08 华南地区的订单数是多少？排除取消订单。


## 3.6 怎样判断升级值得做

先为每个问题标注**必要知识集合**。前 K 条召回率（Recall@K, Recall at K）等于“前 K 条中命中的必要文档数 / 必要文档总数”；还应检查**全部必要文档是否都到齐**。平均倒数排名（MRR, Mean Reciprocal Rank）关注首个相关文档的位置。两者都不直接保证生成的 SQL 正确，后续仍需核对执行结果与最终回答。

例如必要集合为两张卡，而前三条只含一张，则 Recall@3 是 1/2，“全部必需命中@3”为假。先测量基线，再比较改写、扩大候选、重排序带来的准确率、调用数和延迟。

In [7]:
required = {"metric.net_sales", "join.customer_orders"}
ranked = ["table.payments", "metric.net_sales", "table.refunds", "join.customer_orders"]
top3 = set(ranked[:3])
recall_at_3 = len(top3 & required) / len(required)
all_required_at_3 = required <= top3
first_relevant_rank = next(i for i, doc_id in enumerate(ranked, 1) if doc_id in required)
print({"Recall@3": recall_at_3,
       "全部必需命中@3": all_required_at_3,
       "单题倒数排名": 1 / first_relevant_rank})
assert recall_at_3 == 0.5 and not all_required_at_3

{'Recall@3': 0.5, '全部必需命中@3': False, '单题倒数排名': 0.5}


## 练习

**基础练习**：把问题改成“退款后销售额”，对比词法排名与融合排名；再只用 <code>paid_at</code> 提问，解释为什么当前词法通道没有字段级全文匹配。**检查标准**：指出命中的具体别名或 <code>object_ref</code> 片段，不把“正文里有字段”误认为词法命中原因；能手算至少一张卡的 RRF 两项贡献。

**进阶练习**：为“2026 年 8 月华东和华南地区净销售额分别是多少”设计两种检索策略：单次查询和拆成两个子查询。列出每种方案的必要知识 ID、候选数、预期调用次数与可能遗漏。**检查标准**：两地结果都需使用同一指标口径；拆解不得丢掉月份与退款状态；最终用独立的结构化计算验证金额。

**上线判断**：若首轮候选没有 <code>metric.net_sales</code>，重排序能修复吗？不能。先改进召回或知识准备，并记录失败样例。

## 继续阅读

- [pgvector 官方混合检索建议](https://github.com/pgvector/pgvector)
- [Azure 混合检索和 RRF 说明](https://learn.microsoft.com/en-us/azure/search/hybrid-search-overview)
- [项目检索实现](../src/nl2sql/retrieval.py)